### Librerías a utilizar
---

In [7]:
import os, mlflow
import pickle
import pandas as pd
from sklearn.metrics import  root_mean_squared_error
from sklearn.feature_extraction import  DictVectorizer
import os, mlflow
from dotenv import load_dotenv
import math
import optuna
import pathlib
from optuna.samplers import TPESampler
from mlflow.models.signature import infer_signature
from sklearn.ensemble import RandomForestRegressor
from mlflow import MlflowClient
from datetime import datetime
import mlflow.pyfunc as mlflow_pyfunc
from statsmodels.stats.outliers_influence import variance_inflation_factor
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split


### Cargar las credenciales
---

In [2]:
load_dotenv(override=True)  # Carga las variables del archivo .env
EXPERIMENT_NAME = "/Users/sarahbeltrang@gmail.com/project1-experiment" 

mlflow.set_tracking_uri("databricks")
experiment = mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

2025/11/04 21:36:16 INFO mlflow.tracking.fluent: Experiment with name '/Users/sarahbeltrang@gmail.com/project1-experiment' does not exist. Creating a new experiment.


### Preprocessing
___

In [ ]:
def preprocessing(df: pd.DataFrame):
    # Quitar columnas
    df = df.drop(columns=['Health_Issues', 'Caffeine_mg'], errors='ignore')

    # Filtrar filas con género "Other"
    df = df[df["Gender"] != "Other"]

    # Mapear países a continentes 
    pais_a_continente = {
        "Canada": "America", "USA": "America", "Mexico": "America", "Brazil": "America",
        "Norway": "Europe", "Sweden": "Europe", "UK": "Europe", "Finland": "Europe",
        "Italy": "Europe", "Belgium": "Europe", "Germany": "Europe", "France": "Europe",
        "Switzerland": "Europe", "Netherlands": "Europe", "Spain": "Europe",
        "India": "Asia", "China": "Asia", "South Korea": "Asia", "Japan": "Asia",
        "Australia": "Oceania"
    }
    df["Continent"] = df["Country"].map(pais_a_continente)

    # Mapear variables categóricas 
    df['Sleep_Quality'] = df['Sleep_Quality'].map({'Poor': 0, 'Fair': 1, 'Good': 2, 'Excellent': 3})
    df['Stress_Level'] = df['Stress_Level'].map({'Low': 0, 'Medium': 1, 'High': 2})
    df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})

    # Eliminar columnas
    df = df.drop(columns=["Country"], errors='ignore')
    if "ID" in df.columns:
        df = df.drop(columns=["ID"])

    # Convertir booleanos a enteros 
    for col in df.columns:
        if df[col].dtype == 'bool':
            df[col] = df[col].astype(int)

    # Convertir a matriz con DictVectorizer 
    dicts = df.drop(columns=["Stress_Level"]).to_dict(orient="records")
    dv = DictVectorizer(sparse=False)
    X = dv.fit_transform(dicts)

    # Target 
    y = df["Stress_Level"].values

    # Balancear con SMOTE 
    smote = SMOTE(random_state=42)
    X_bal, y_bal = smote.fit_resample(X, y)

    print("Preprocessing completado.")
    print("Shape features balanceadas:", X_bal.shape)
    print("Distribución target balanceado:\n", pd.Series(y_bal).value_counts())

    return X_bal, y_bal, dv


In [ ]:
df = pd.read_csv("../data/raw/synthetic_coffee_health_10000.csv")

In [10]:
target = 'Stress_Level'  
X = df.drop(columns=[target])
y = df[target].values

### Dividir en entrenamiento, prueba & validacion
---

In [11]:
X_train, X_test_val, y_train, y_test_val = train_test_split(
    X, y, test_size=0.4, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_test_val, y_test_val, test_size=0.5, random_state=42
)

### Regresion Logistica
---

#### Funcion objetivo